In [1]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.1.1+cu118 requires torch==2.1.1, but you have torch 2.10.0 which is incompatible.
torchtext 0.16.1+cpu requires torch==2.1.1, but you have torch 2.10.0 which is incompatible.
torchvision 0.16.1+cu118 requires torch==2.1.1, but you have torch 2.10.0 which is incompatible.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 6454
LORA_RANK = 8

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

enable_thinking = False

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_json("../data/preprocessed/train.jsonl", lines=True)
test_df  = pd.read_json("../data/preprocessed/test.jsonl",  lines=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Columns: {list(train_df.columns)}")
train_df.head()

In [ ]:
def format_chat(row):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["ideal_response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=enable_thinking)


train_df["text"] = train_df.apply(format_chat, axis=1)
dataset = Dataset.from_pandas(train_df[["text"]])

print(f"Example length (chars): {len(dataset[0]['text'])}")
print(dataset[0]["text"][:500])

In [ ]:
lengths = [len(tokenizer.encode(t)) for t in train_df["text"]]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)//len(lengths)}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"Примеров влезающих полностью: {sum(1 for l in lengths if l <= MAX_SEQ_LENGTH)}/{len(lengths)}")

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
        gradient_checkpointing=True,
    ),
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


In [ ]:
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
train_logs = logs[logs["loss"].notna()] if "loss" in logs.columns else logs

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], marker="o", color="#d62728")
axes[0].set_title("Training loss")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[0].grid(alpha=0.3)

if "learning_rate" in train_logs.columns:
    axes[1].plot(train_logs["step"], train_logs["learning_rate"], marker="o", color="#1f77b4")
    axes[1].set_title("Learning rate")
    axes[1].set_xlabel("step"); axes[1].set_ylabel("lr")
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
model.save_pretrained("koyash-qwen3-4b-lora")
tokenizer.save_pretrained("koyash-qwen3-4b-lora")

In [ ]:
from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "koyash-qwen3-4b-lora",
    torch_dtype=torch.float16,
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("koyash-qwen3-4b-merged")
tokenizer.save_pretrained("koyash-qwen3-4b-merged")